In [1]:
# 🛠️ Install necessary libraries
!pip install -U transformers

In [2]:
!pip install -q transformers datasets

In [3]:
import pandas as pd
import torch
from torch.utils.data import Dataset
from sklearn.metrics import accuracy_score
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments

In [4]:

# Load data
train_df = pd.read_csv("/content/input_train_annot_t5.csv").dropna()
val_df = pd.read_csv("/content/input_val_annot_t5.csv").dropna()
test_df = pd.read_csv("/content/input_test_annot_t5.csv").dropna()



In [5]:
# Dataset Class
class QADataset(Dataset):
    def __init__(self, df, tokenizer, max_input_len=512, max_target_len=64):
        self.tokenizer = tokenizer
        self.inputs = df["input"].tolist()
        self.targets = df["output"].tolist()
        self.max_input_len = max_input_len
        self.max_target_len = max_target_len

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        input_enc = self.tokenizer(
            self.inputs[idx],
            max_length=self.max_input_len,
            truncation=True,
            padding='max_length',
            return_tensors="pt"
        )

        target_enc = self.tokenizer(
            self.targets[idx],
            max_length=self.max_target_len,
            truncation=True,
            padding='max_length',
            return_tensors="pt"
        )

        labels = target_enc["input_ids"].squeeze()
        labels[labels == tokenizer.pad_token_id] = -100  # Ignore padding in loss

        return {
            "input_ids": input_enc["input_ids"].squeeze(),
            "attention_mask": input_enc["attention_mask"].squeeze(),
            "labels": labels
        }



In [6]:
# Load tokenizer and model
model_name = "t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [7]:
# Create datasets
train_dataset = QADataset(train_df, tokenizer)
val_dataset = QADataset(val_df, tokenizer)
test_dataset = QADataset(test_df, tokenizer)

In [8]:
# Training arguments
training_args = TrainingArguments(
    output_dir="./t5-model",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=6,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="none"
)

In [9]:
# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

#Train the model
trainer.train()


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss
1,1.050000,0.891617
2,0.886600,0.806102
3,0.814300,0.763809
4,0.784400,0.752241
5,0.751400,0.750637
6,0.717600,0.749933


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=7422, training_loss=0.8501517795160818, metrics={'train_runtime': 4700.2427, 'train_samples_per_second': 25.261, 'train_steps_per_second': 1.579, 'total_flos': 1.6069673455976448e+16, 'train_loss': 0.8501517795160818, 'epoch': 6.0})

In [10]:
# Save the model and tokenizer
model.save_pretrained("./t5-model-final")
tokenizer.save_pretrained("./t5-model-final")


('./t5-model-final/tokenizer_config.json',
 './t5-model-final/special_tokens_map.json',
 './t5-model-final/spiece.model',
 './t5-model-final/added_tokens.json')

In [14]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [18]:
import shutil

# Save to Drive
drive_path = "/content/drive/MyDrive/t5-model-final"
shutil.copytree("./t5-model-final", drive_path)
print(f" Model copied to Google Drive: {drive_path}")


FileExistsError: [Errno 17] File exists: '/content/drive/MyDrive/t5-model-final'

In [19]:
def evaluate(model, dataset, tokenizer, df_ref, output_csv="predictions.csv"):
    model.eval()
    preds, targets = [], []

    for i, sample in enumerate(dataset):
        input_ids = sample["input_ids"].unsqueeze(0).to(model.device)
        attention_mask = sample["attention_mask"].unsqueeze(0).to(model.device)

        with torch.no_grad():
            output_ids = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_length=64,
                num_beams=4,
                early_stopping=True
            )[0]

        pred = tokenizer.decode(output_ids, skip_special_tokens=True)
        label_ids = sample["labels"]
        label_ids = label_ids[label_ids != -100]
        target = tokenizer.decode(label_ids, skip_special_tokens=True)

        preds.append(pred.strip())
        targets.append(target.strip())

    exact_matches = [p == t for p, t in zip(preds, targets)]
    accuracy = sum(exact_matches) / len(exact_matches) * 100
    print(f"\n🎯 Exact Match Accuracy: {accuracy:.2f}%")

    # Sample output
    for i in range(min(10, len(preds))):
        print(f"\nQ: {df_ref['input'].iloc[i]}")
        print(f"Predicted: {preds[i]}")
        print(f"Ground Truth: {targets[i]}")

    # Save to CSV
    output_df = pd.DataFrame({
        "Input": df_ref["input"],
        "Prediction": preds,
        "Ground Truth": targets,
        "Match": exact_matches
    })
    output_df.to_csv(output_csv, index=False)
    print(f"\n📁 Saved predictions to: {output_csv}")


In [20]:
evaluate(model, val_dataset, tokenizer, val_df, output_csv="val_predictions.csv")



🎯 Exact Match Accuracy: 65.25%

Q: question: What was the global sales income of certified Fairtrade International coffee in 2018? table: Certified product: Coffee, Sales income in million euros: 76.4; Certified product: Cocoa, Sales income in million euros: 44.4; Certified product: Bananas, Sales income in million euros: 32.2; Certified product: Cane sugar, Sales income in million euros: 10.7; Certified product: Flowers and plants, Sales income in million euros: 6.7; Certified product: Tea, Sales income in million euros: 4.7; Certified product: Cotton, Sales income in million euros: 1.4; chart_type: h_bar title:  x_axis_title: ['Coffee', 'Cocoa', 'Bananas', 'Cane sugar', 'Flowers and plants', 'Tea', 'Cotton'] y_axis_title: Sales income in million euros
Predicted: 76.4
Ground Truth: 76.4

Q: question: In what year did the Mexican government spend 6.61 billion dollars in the military? table: Characteristic: 2020, Expenditure in billion U.S. dollars: 6.61; Characteristic: 2019, Expendit

In [21]:
evaluate(model, test_dataset, tokenizer, test_df, output_csv="test_predictions.csv")



🎯 Exact Match Accuracy: 66.25%

Q: question: What game topped the charts with 512.3 million hours watched on Twitch in the first half of 2019? table: Characteristic: League of Legends, Number of hours watched: 512.3; Characteristic: Fortnite, Number of hours watched: 465.0; Characteristic: Just Chatting, Number of hours watched: 372.5; Characteristic: Grand Theft Auto V, Number of hours watched: 269.1; Characteristic: Dota 2, Number of hours watched: 237.1; Characteristic: Apex Legends, Number of hours watched: 181.4; Characteristic: Counter-Strike: Global Offensive, Number of hours watched: 178.0; Characteristic: Overwatch, Number of hours watched: 127.3; Characteristic: Hearthstone, Number of hours watched: 120.7; Characteristic: World of Warcraft, Number of hours watched: 118.5; chart_type: h_bar title:  x_axis_title: ['League of Legends', 'Fortnite', 'Just Chatting', 'Grand Theft Auto V', 'Dota 2', 'Apex Legends', 'Counter-Strike: Global Offensive', 'Overwatch', 'Hearthstone', 'Wo